In [ ]:
!pip install pyhealth

In [ ]:
import tempfile

from pyhealth.datasets import MIMIC3Dataset
from pyhealth.datasets import split_by_patient, get_dataloader
from pyhealth.models import Transformer
from pyhealth.tasks import DrugRecommendationMIMIC3
from pyhealth.trainer import Trainer
import torch

from google.colab import drive
drive.mount('/content/drive')

if __name__ == "__main__":
    # STEP 1: load data
    base_dataset = MIMIC3Dataset(
        root="/content/drive/MyDrive/mimic-iii-clinical-database-1.4",
        tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
        cache_dir=tempfile.TemporaryDirectory().name,
        dev=False,
    )
    base_dataset.stats()

    # STEP 2: set task
    task = DrugRecommendationMIMIC3()
    print(task.input_schema)
    print(task.output_schema)
    sample_dataset = base_dataset.set_task(task)

    train_dataset, val_dataset, test_dataset = split_by_patient(
        sample_dataset, [0.8, 0.1, 0.1]
    )
    train_dataloader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
    val_dataloader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
    test_dataloader = get_dataloader(test_dataset, batch_size=32, shuffle=False)

    # STEP 3: define model
    model = Transformer(
        dataset=sample_dataset,
    )

    # STEP 4: define trainer
    trainer = Trainer(
        model=model,
        metrics=["jaccard_samples", "f1_samples", "pr_auc_samples"],
    )

    trainer.train(
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        epochs=1,
        monitor="pr_auc_samples",
    )

    # STEP 5: evaluate
    print(trainer.evaluate(test_dataloader))

    #precision@k and recall@k metrics
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch in test_dataloader:
            output = model(**batch)

            y_true.append(output["y_true"].cpu())
            y_pred.append(output["y_prob"].cpu())

    y_true = torch.cat(y_true)
    y_pred = torch.cat(y_pred)

    def precision_at_k(y_true, y_pred, k):
      topk = torch.topk(y_pred, k, dim=1).indices

      precisions = []

      for i in range(y_true.shape[0]):
          true_set = set(torch.where(y_true[i] == 1)[0].tolist())
          pred_set = set(topk[i].tolist())

          precisions.append(len(true_set & pred_set) / k)

      return sum(precisions) / len(precisions)

    def recall_at_k(y_true, y_pred, k):
      topk = torch.topk(y_pred, k, dim=1).indices

      recalls = []

      for i in range(y_true.shape[0]):
          true_set = set(torch.where(y_true[i] == 1)[0].tolist())
          pred_set = set(topk[i].tolist())

          if len(true_set) == 0:
              recalls.append(0)
          else:
              recalls.append(len(true_set & pred_set) / len(true_set))

      return sum(recalls) / len(recalls)

    print("Precision@10:", precision_at_k(y_true, y_pred, 10))
    print("Recall@10:", recall_at_k(y_true, y_pred, 10))

    print("Precision@20:", precision_at_k(y_true, y_pred, 20))
    print("Recall@20:", recall_at_k(y_true, y_pred, 20))